# Lesson 4A: Gaussian Mixture Models - Theory

## Introduction

**Gaussian Mixture Models (GMM)** represent data as a mixture of multiple Gaussian distributions. Unlike K-Means which assigns each point to exactly one cluster, GMM provides **soft assignments** - probabilities of belonging to each cluster.

**Applications**:
- Soft clustering with uncertainty
- Density estimation
- Anomaly detection
- Image segmentation

**Key Idea**: Data is generated from a mixture of K Gaussians:

$$p(x) = \sum_{k=1}^{K} \pi_k \mathcal{N}(x | \mu_k, \Sigma_k)$$

We learn parameters $\{\pi_k, \mu_k, \Sigma_k\}$ using the **EM Algorithm**.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import make_blobs, make_moons
from sklearn.preprocessing import StandardScaler
from typing import Tuple
from numpy.typing import NDArray

np.random.seed(42)
plt.style.use('seaborn-v0_8-darkgrid')

print("✅ Libraries loaded!")

## EM Algorithm

**E-Step**: Compute responsibilities (soft assignments):

$$\gamma_{ik} = \frac{\pi_k \mathcal{N}(x_i | \mu_k, \Sigma_k)}{\sum_{j=1}^{K} \pi_j \mathcal{N}(x_i | \mu_j, \Sigma_j)}$$

**M-Step**: Update parameters:

$$\pi_k = \frac{1}{N} \sum_{i=1}^{N} \gamma_{ik}$$

$$\mu_k = \frac{\sum_{i=1}^{N} \gamma_{ik} x_i}{\sum_{i=1}^{N} \gamma_{ik}}$$

$$\Sigma_k = \frac{\sum_{i=1}^{N} \gamma_{ik} (x_i - \mu_k)(x_i - \mu_k)^T}{\sum_{i=1}^{N} \gamma_{ik}}$$

In [ ]:
from scipy.stats import multivariate_normal

class GMM:
    """Gaussian Mixture Model with EM algorithm."""
    
    def __init__(self, n_components=3, max_iters=100):
        self.n_components = n_components
        self.max_iters = max_iters
        self.weights = None
        self.means = None
        self.covariances = None
        
    def fit(self, X: NDArray):
        n_samples, n_features = X.shape
        
        # Initialize parameters randomly
        self.weights = np.ones(self.n_components) / self.n_components
        random_idx = np.random.choice(n_samples, self.n_components, replace=False)
        self.means = X[random_idx]
        self.covariances = [np.eye(n_features) for _ in range(self.n_components)]
        
        for iteration in range(self.max_iters):
            # E-step: Compute responsibilities
            responsibilities = self._e_step(X)
            
            # M-step: Update parameters
            self._m_step(X, responsibilities)
            
        return self
    
    def _e_step(self, X: NDArray) -> NDArray:
        """Compute responsibilities (soft assignments)."""
        n_samples = X.shape[0]
        responsibilities = np.zeros((n_samples, self.n_components))
        
        for k in range(self.n_components):
            responsibilities[:, k] = self.weights[k] * \
                multivariate_normal.pdf(X, self.means[k], self.covariances[k])
        
        # Normalize
        responsibilities /= responsibilities.sum(axis=1, keepdims=True)
        return responsibilities
    
    def _m_step(self, X: NDArray, responsibilities: NDArray):
        """Update parameters."""
        n_samples = X.shape[0]
        
        for k in range(self.n_components):
            resp_k = responsibilities[:, k]
            total_resp = resp_k.sum()
            
            # Update weight
            self.weights[k] = total_resp / n_samples
            
            # Update mean
            self.means[k] = (resp_k[:, np.newaxis] * X).sum(axis=0) / total_resp
            
            # Update covariance
            diff = X - self.means[k]
            self.covariances[k] = (resp_k[:, np.newaxis] * diff).T @ diff / total_resp
            self.covariances[k] += np.eye(X.shape[1]) * 1e-6  # Regularization
    
    def predict_proba(self, X: NDArray) -> NDArray:
        """Predict soft cluster assignments."""
        return self._e_step(X)
    
    def predict(self, X: NDArray) -> NDArray:
        """Predict hard cluster assignments."""
        return self.predict_proba(X).argmax(axis=1)

print("✅ GMM with EM algorithm implemented!")

In [ ]:
# Test GMM
X, y = make_blobs(n_samples=300, centers=3, cluster_std=0.6, random_state=42)

gmm = GMM(n_components=3, max_iters=50)
gmm.fit(X)
labels = gmm.predict(X)

plt.figure(figsize=(10, 6))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', edgecolors='k', s=50)
plt.scatter(gmm.means[:, 0], gmm.means[:, 1], c='red', marker='X', 
            s=300, edgecolors='black', linewidths=2, label='Centroids')
plt.title('GMM Clustering', fontsize=14, fontweight='bold')
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print("✅ GMM successfully clustered data with soft assignments!")